In [20]:
import json
import sys
from pathlib import Path

# Enable autoreload so notebook picks up local code changes without kernel restart.
from IPython import get_ipython
ip = get_ipython()
if ip is not None:
    ip.run_line_magic("load_ext", "autoreload")
    ip.run_line_magic("autoreload", "2")

# Resolve project root and ensure local package imports work from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / "nepse_analyst").exists():
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BENCHMARK_PATH = PROJECT_ROOT / "evaluation" / "benchmark_questions.json"
REPORTS_DIR = PROJECT_ROOT / "evaluation" / "results"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

from nepse_analyst.sql_generator import generate_and_execute

In [11]:
with open(BENCHMARK_PATH) as f:
    benchmark = json.load(f)

questions = benchmark["sql_benchmark"]
print(f"Loaded {len(questions)} benchmark questions")

Loaded 12 benchmark questions


In [12]:
results = []

for q in questions:
    print(f"\n{'='*60}")
    print(f"[{q['id']}] {q['question']}")
    
    result = generate_and_execute(q['question'])
    
    print(f"SQL generated (attempt {result['attempts']}):")
    print(result['sql'])
    print(f"Rows returned: {result['row_count']}")
    if result['rows']:
        print(f"First row: {result['rows'][0]}")
    if not result['success']:
        print(f"ERROR: {result['error']}")
    
    results.append({
        "id": q['id'],
        "question": q['question'],
        "difficulty": q['difficulty'],
        "ground_truth": q['ground_truth'],
        "generated_sql": result['sql'],
        "returned_rows": result['rows'][:3],  # save first 3 rows
        "row_count": result['row_count'],
        "attempts": result['attempts'],
        "success": result['success'],
        "error": result.get('error'),
        "score": None,  # to be filled manually below
        "notes": ""
    })


[Q1] Which commercial bank has the highest EPS in the latest fiscal year?
SQL generated (attempt 1):
SELECT c.symbol, c.name, f.eps, f.fiscal_year FROM fundamentals f JOIN companies c ON f.symbol = c.symbol WHERE c.sector = 'Commercial Banks' AND f.fiscal_year = (SELECT MAX(fiscal_year) FROM fundamentals WHERE symbol IN (SELECT symbol FROM companies WHERE sector = 'Commercial Banks')) ORDER BY f.eps DESC LIMIT 1
Rows returned: 1
First row: {'symbol': 'NABIL', 'name': 'Nabil Bank Limited', 'eps': 35.18, 'fiscal_year': '2082/83'}

[Q2] List all hydropower companies that have paid cash dividends for 3 or more consecutive years.
SQL generated (attempt 1):
SELECT c.symbol, c.name, d.fiscal_year, d.cash_dividend 
FROM dividends d JOIN companies c ON d.symbol = c.symbol 
WHERE c.sector = 'Hydropower' 
GROUP BY c.symbol, c.name, d.fiscal_year, d.cash_dividend 
HAVING COUNT(DISTINCT d.fiscal_year) >= 3
Rows returned: 0

[Q3] What is the P/E ratio comparison of the top 5 commercial banks by mar

In [13]:
# Fill in scores manually after reviewing each result above
scores = {
    "Q1":  1.0,   # correct — NABIL with correct EPS
    "Q2":  0.5,   # partial — right companies but missing one
    "Q3":  1.0,
    # ... fill in all 12
}

for r in results:
    r['score'] = scores.get(r['id'], 0.0)

total = sum(r['score'] for r in results)
print(f"\nTotal score: {total:.1f} / {len(results)}.0")
print(f"Accuracy: {total/len(results)*100:.1f}%")

# By difficulty
for diff in ['easy', 'medium', 'hard']:
    subset = [r for r in results if r['difficulty'] == diff]
    s = sum(r['score'] for r in subset)
    print(f"  {diff.capitalize()}: {s:.1f}/{len(subset)}.0")


Total score: 2.5 / 12.0
Accuracy: 20.8%
  Easy: 1.0/4.0
  Medium: 1.5/6.0
  Hard: 0.0/2.0


In [14]:
from datetime import datetime

report = {
    "run_date": datetime.now().isoformat(),
    "total_score": total,
    "max_score": len(results),
    "accuracy_pct": round(total / len(results) * 100, 1),
    "pass_threshold": 8.0,
    "passed": total >= 8.0,
    "results": results,
}

report_path = REPORTS_DIR / f"eval_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {report_path}")

Report saved to /home/meutsabdahal/Desktop/nepse-analyst/evaluation/results/eval_20260406_0929.json


In [ ]:
from time import perf_counter
from nepse_analyst.retriever import search
from nepse_analyst.prompts import build_rag_synthesis_prompt
from nepse_analyst import llm
from nepse_analyst.language_detector import detect_language

rag_questions = [
    {"id": "R1", "question": "What recent news is there about Nabil Bank?"},
    {"id": "R2", "question": "Are there any upcoming AGMs announced in the banking sector?"},
    {"id": "R3", "question": "What is the regulatory environment for hydropower IPOs in Nepal currently?"},
    {"id": "R4", "question": "Summarise the latest quarterly earnings news for major commercial banks."},
    {"id": "R5", "question": "कुन कम्पनीले हालसालै बोनस सेयर घोषणा गर्यो?"},
]

rag_expectations = {
    "R1": ["nabil", "bank"],
    "R2": ["agm", "bank"],
    "R3": ["hydropower", "ipo", "regulat", "sebon"],
    "R4": ["quarter", "earnings", "bank"],
    "R5": ["bonus", "बोनस", "share", "सेयर"],
}

rag_results = []

for q in rag_questions:
    print(f"\n{'='*60}\n[{q['id']}] {q['question']}")
    
    t0 = perf_counter()
    passages = search(q['question'], top_k=5)
    retrieval_ms = (perf_counter() - t0) * 1000
    query_lang = detect_language(q['question'])
    
    print(f"Top {len(passages)} passages retrieved in {retrieval_ms:.1f} ms:")
    for p in passages:
        print(f"  [{p['relevance_score']:.3f}] {p['published_at']} | {p['title'][:70]}")
    
    top_passage_relevant = False
    if passages:
        prompt = build_rag_synthesis_prompt(q['question'], passages, query_lang)
        try:
            answer = llm.call(prompt, temperature=0.1)
        except Exception as exc:
            answer = f"Synthesis unavailable: {exc}"
        print(f"\nSynthesised answer:\n{answer[:400]}")

        blob = f"{passages[0].get('title', '')} {passages[0].get('content', '')}".lower()
        expected_terms = rag_expectations.get(q['id'], [])
        top_passage_relevant = any(term.lower() in blob for term in expected_terms) if expected_terms else True
    else:
        answer = "No relevant passages found in the corpus."
        print("No passages retrieved.")
    
    rag_results.append({
        "id": q['id'],
        "question": q['question'],
        "query_language": query_lang,
        "latency_ms": round(retrieval_ms, 2),
        "passages_retrieved": len(passages),
        "top_passage_title": passages[0]['title'] if passages else "",
        "top_relevance_score": passages[0]['relevance_score'] if passages else 0.0,
        "synthesised_answer": answer,
        "top_passage_relevant": top_passage_relevant,
        "notes": ""
    })

rag_relevance = sum(1 for r in rag_results if r['top_passage_relevant']) / max(len(rag_results), 1)
rag_coverage = sum(1 for r in rag_results if r['passages_retrieved'] > 0) / max(len(rag_results), 1)
print(f"\nRAG top-passage relevance: {rag_relevance*100:.1f}%")
print(f"RAG retrieval coverage: {rag_coverage*100:.1f}%")


[R1] What recent news is there about Nabil Bank?
Top 5 passages retrieved:
  [0.705] 2016-12-15 | The Nilaya Hotels Holds First AGM, Plans IPO and Strategic Expansion
  [0.698] 2016-12-15 | Capital plan of Commercial Bank
  [0.666] 2016-12-15 | Daraz Nepal Launches 4.4 Nawa Barsha Utsav with Mega Deals and Giveawa
  [0.637] 2016-12-15 | SEBON Approves Margin Trading: Investors Can Now Buy Stocks With Broke
  [0.603] 2016-12-15 | Mahalaxmi Bikas Bank Launches Digital Calendar 2083

Synthesised answer:
There is recent news about Nabil Bank. According to passage [3], Nabil Bank is one of the four leading banks that are offering additional prepayment discounts during Daraz Nepal's 4.4 Nawa Barsha Utsav online shopping festival.

[R2] Are there any upcoming AGMs announced in the banking sector?
Top 5 passages retrieved:
  [0.692] 2016-12-15 | The Nilaya Hotels Holds First AGM, Plans IPO and Strategic Expansion
  [0.690] 2016-12-15 | Capital plan of Commercial Bank
  [0.641] 2016-12-15 | Go

In [21]:
import importlib
import nepse_analyst.router as router_mod
import nepse_analyst.retriever as retriever_mod
import nepse_analyst.llm as llm_mod
import nepse_analyst.sql_generator as sql_gen_mod
import nepse_analyst.pipeline as pipeline_mod

# Force reload of local modules so this notebook always uses latest code edits.
for mod in [router_mod, retriever_mod, llm_mod, sql_gen_mod, pipeline_mod]:
    importlib.reload(mod)

run = pipeline_mod.run

# The 4 out-of-scope queries — all must be declined
oos_queries = [
    ("X1", "Will NABIL stock go up tomorrow?"),
    ("X2", "Should I buy or sell HIDCL right now?"),
    ("X3", "Which is the best stock for quick returns?"),
    ("X4", "What will the NEPSE index be at the end of this year?"),
]
print("=== OUT-OF-SCOPE GUARDRAIL TESTS ===")
for qid, query in oos_queries:
    result = run(query)
    status = "DECLINED" if result["route"] == "OOS" else "FAILED TO DECLINE"
    print(f"[{qid}] {status} | route={result['route']} | {query[:50]}")

# The 12 SQL benchmark questions
sql_queries = [
    ("Q1",  "Which commercial bank has the highest EPS in the latest fiscal year?"),
    ("Q2",  "List all hydropower companies that have paid cash dividends for 3 or more consecutive years."),
    ("Q3",  "What is the P/E ratio comparison of the top 5 commercial banks by market capitalisation?"),
    ("Q4",  "Which sector has had the highest average daily turnover over the last 30 trading days?"),
    ("Q5",  "Which companies have shown consistent EPS growth for 3 consecutive fiscal years?"),
    ("Q6",  "What are the top 5 most oversubscribed IPOs in NEPSE history?"),
    ("Q7",  "Compare total bonus shares issued by the banking sector vs. the hydropower sector in FY 2080/81."),
    ("Q8",  "Which company currently has the highest book value per share?"),
    ("Q9",  "What is the 52-week high and low for NABIL Bank?"),
    ("Q10", "Which microfinance company has the highest ROE in the latest fiscal year?"),
    ("Q11", "How has the NEPSE Banking Index performed compared to the main NEPSE index over the past year?"),
    ("Q12", "Which companies had an IPO listing price at least 50% above their issue price in the last 3 years?"),
]
print("\n=== SQL BENCHMARK ROUTING CHECK ===")
for qid, query in sql_queries:
    result = run(query)
    routed_correctly = result["route"] in ("SQL", "HYBRID")
    status = "✅" if routed_correctly else f"⚠️  route={result['route']}"
    print(f"[{qid}] {status} | success={result['success']} | {query[:60]}")
    if result.get("sql"):
        print(f"       SQL: {result['sql'][:80]}...")

# The 5 RAG benchmark questions 
rag_queries = [
    ("R1", "What recent news is there about Nabil Bank?"),
    ("R2", "Are there any upcoming AGMs announced in the banking sector?"),
    ("R3", "What is the regulatory environment for hydropower IPOs in Nepal currently?"),
    ("R4", "Summarise the latest quarterly earnings news for major commercial banks."),
    ("R5", "कुन कम्पनीले हालसालै बोनस सेयर घोषणा गर्यो?"),
]
print("\n=== RAG BENCHMARK ROUTING CHECK ===")
for qid, query in rag_queries:
    result = run(query)
    routed_correctly = result["route"] in ("RAG", "HYBRID")
    status = "✅" if routed_correctly else f"⚠️  route={result['route']}"
    print(f"[{qid}] {status} | passages={len(result.get('passages', []))} | {query[:60]}")
    if result.get("passages"):
        print(f"       Top passage: {result['passages'][0]['title'][:70]}")

=== OUT-OF-SCOPE GUARDRAIL TESTS ===
[X1] DECLINED | route=OOS | Will NABIL stock go up tomorrow?
[X2] DECLINED | route=OOS | Should I buy or sell HIDCL right now?
[X3] DECLINED | route=OOS | Which is the best stock for quick returns?
[X4] DECLINED | route=OOS | What will the NEPSE index be at the end of this ye

=== SQL BENCHMARK ROUTING CHECK ===
[Q1] ✅ | success=True | Which commercial bank has the highest EPS in the latest fisc
       SQL: SELECT c.symbol, c.name, f.eps, f.fiscal_year FROM fundamentals f JOIN companies...
[Q2] ✅ | success=True | List all hydropower companies that have paid cash dividends 
       SQL: SELECT c.symbol, c.name, d.fiscal_year, d.cash_dividend 
FROM dividends d JOIN c...
[Q3] ✅ | success=True | What is the P/E ratio comparison of the top 5 commercial ban
       SQL: SELECT c.symbol, c.name, c.market_cap, f.pe_ratio, f.fiscal_year FROM fundamenta...
[Q4] ✅ | success=True | Which sector has had the highest average daily turnover over
       SQL: SELECT c.

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[R1] ✅ | passages=1 | What recent news is there about Nabil Bank?
       Top passage: Daraz Nepal Launches 4.4 Nawa Barsha Utsav with Mega Deals and Giveawa
[R2] ✅ | passages=2 | Are there any upcoming AGMs announced in the banking sector?
       Top passage: Himalayan Bank to Convert 10% Promoter Shares into Public Shares
[R3] ✅ | passages=5 | What is the regulatory environment for hydropower IPOs in Ne
       Top passage: Machhapuchchhre Capital Appointed Issue Manager for Nepal Ceramic IPO
[R4] ✅ | passages=2 | Summarise the latest quarterly earnings news for major comme
       Top passage: Daraz Nepal Launches 4.4 Nawa Barsha Utsav with Mega Deals and Giveawa
[R5] ✅ | passages=5 | कुन कम्पनीले हालसालै बोनस सेयर घोषणा गर्यो?
       Top passage: See how much bonus &amp; right shares Insurance companies will be issu


In [ ]:
import importlib
import json
import time
from datetime import datetime
import nepse_analyst.router as router_mod
import nepse_analyst.retriever as retriever_mod
import nepse_analyst.llm as llm_mod
import nepse_analyst.sql_generator as sql_gen_mod
import nepse_analyst.pipeline as pipeline_mod

# Force reload of local modules so this notebook always uses latest code edits.
for mod in [router_mod, retriever_mod, llm_mod, sql_gen_mod, pipeline_mod]:
    importlib.reload(mod)

run = pipeline_mod.run

all_queries = oos_queries + sql_queries + rag_queries

integration_results = []
for qid, query in all_queries:
    t0 = time.perf_counter()
    result = run(query)
    latency_ms = (time.perf_counter() - t0) * 1000
    
    category = "OOS" if qid.startswith("X") else \
               "SQL" if qid.startswith("Q") else "RAG"
    
    # For OOS: pass = route is OOS. For SQL/RAG: pass = correct route and success
    if category == "OOS":
        passed = result["route"] == "OOS"
    elif category == "SQL":
        passed = result["route"] in ("SQL", "HYBRID") and result["success"]
    else:
        passed = result["route"] in ("RAG", "HYBRID") and len(result.get("passages", [])) > 0
    
    integration_results.append({
        "id": qid,
        "category": category,
        "query": query,
        "route": result["route"],
        "success": result["success"],
        "passed": passed,
        "latency_ms": round(latency_ms, 2),
        "answer_preview": result["answer"][:200] if result.get("answer") else "",
        "error": result.get("error")
    })

total_passed = sum(1 for r in integration_results if r["passed"] )
print(f"\nIntegration Test Results: {total_passed}/21 passed")
print(f"OOS: {sum(1 for r in integration_results if r['category']=='OOS' and r['passed'])}/4")
print(f"SQL: {sum(1 for r in integration_results if r['category']=='SQL' and r['passed'])}/12")
print(f"RAG: {sum(1 for r in integration_results if r['category']=='RAG' and r['passed'])}/5")

latencies = sorted(r['latency_ms'] for r in integration_results)
p50 = latencies[max(0, int((len(latencies) - 1) * 0.50))] if latencies else 0.0
p95 = latencies[max(0, int((len(latencies) - 1) * 0.95))] if latencies else 0.0
max_latency = max(latencies) if latencies else 0.0
under_8s_pct = (sum(1 for x in latencies if x < 8000) / max(len(latencies), 1)) * 100
print(f"Latency p50: {p50:.1f} ms | p95: {p95:.1f} ms | max: {max_latency:.1f} ms")
print(f"Latency under 8s: {under_8s_pct:.1f}%")

# Save
report = {
    "run_date": datetime.now().isoformat(),
    "total_passed": total_passed,
    "max": 21,
    "latency_summary": {
        "p50_ms": round(p50, 2),
        "p95_ms": round(p95, 2),
        "max_ms": round(max_latency, 2),
        "under_8s_pct": round(under_8s_pct, 2),
    },
    "results": integration_results
}
report_path = REPORTS_DIR / f"integration_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {report_path}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given



Integration Test Results: 21/21 passed
OOS: 4/4
SQL: 12/12
RAG: 5/5
Report saved to /home/meutsabdahal/Desktop/nepse-analyst/evaluation/results/integration_20260406_0936.json
